In [ ]:
import os, numpy as np, pandas as pd, time
SH    = '/kaggle/input/datasets/abhiroopsarkar/knee-shards-all/shards'
LAB   = '/kaggle/input/datasets/abhiroopsarkar/knee-labels/report_labels.csv'
TRAIN = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv'
TMP   = '/kaggle/temp/xs'; os.makedirs(TMP, exist_ok=True)
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

uids, masks = [], []
for ci in range(18):
    d = np.load(f'{SH}/shard_{ci:04d}.npz')
    if not os.path.exists(f'{TMP}/x_{ci:04d}.npy'):
        np.save(f'{TMP}/x_{ci:04d}.npy', d['x'])
    uids += list(d['uid']); masks.append(d['m'])
    print(ci, time.strftime('%H:%M:%S'), flush=True)
MASK = np.concatenate(masks).astype(np.float32)          # (4407, 3)

lab = pd.read_csv(LAB).set_index('StudyInstanceUID').loc[uids]    # align by UID
trn = pd.read_csv(TRAIN).set_index('StudyInstanceUID').loc[uids]
Y    = lab[LABELS].values.astype(np.float32)                      # Gemini soft labels
G    = trn[LABELS].values                                         # gold (NaN if none)
gold = ~np.isnan(G).any(1)
print(len(uids), 'studies | gold:', gold.sum())

In [ ]:
import timm, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

BACKBONE, KU, EPOCHS, BS, LR = 'efficientnet_b0', 16, 7, 4, 3e-4
SMOKE = 0            # set e.g. 200 for a quick test run, 0 for full
dev = 'cuda'
torch.backends.cudnn.benchmark = True

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train, s.mm = idx, train, {}
    def __len__(s): return len(s.idx)
    def __getitem__(s, j):
        i = s.idx[j]; ci = i // 256
        if ci not in s.mm: s.mm[ci] = np.load(f'{TMP}/x_{ci:04d}.npy', mmap_mode='r')
        sl = (np.sort(np.random.choice(24, KU, replace=False)) if s.train
              else np.linspace(0, 23, KU).round().astype(int))
        x = torch.from_numpy(np.ascontiguousarray(s.mm[ci][i % 256][:, sl]))   # (3,KU,S,S) uint8
        return x, torch.from_numpy(MASK[i]), torch.from_numpy(Y[i])

class Net(nn.Module):
    def __init__(s, pretrained=True):
        super().__init__()
        s.bb = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0, in_chans=1)
        D = s.bb.num_features
        s.att = nn.ModuleList([nn.Linear(D, 1) for _ in range(3)])
        s.drop = nn.Dropout(0.3)
        s.head = nn.Linear(3 * D + 3, 12)
    def forward(s, x, m):                       # x (B,3,K,H,W) float, m (B,3)
        B, P, K, H, W = x.shape
        f = s.bb(x.reshape(B * P * K, 1, H, W)).float().reshape(B, P, K, -1)
        e = []
        for p in range(3):
            w = torch.softmax(s.att[p](f[:, p]).squeeze(-1), 1)
            e.append((w.unsqueeze(-1) * f[:, p]).sum(1))
        e = torch.stack(e, 1) * m.unsqueeze(-1)  # zero out missing planes
        return s.head(s.drop(torch.cat([e.flatten(1), m], 1)))

def prep(x): return (x.float() / 255 - 0.2) / 0.25
def aug(x):
    B = x.shape[0]
    x = x * (1 + 0.15 * torch.randn(B, 1, 1, 1, 1, device=x.device)) + 0.15 * torch.randn(B, 1, 1, 1, 1, device=x.device)
    dy, dx = np.random.randint(-10, 11, 2)       # no flips: medial/lateral must stay put
    return torch.roll(x, (int(dy), int(dx)), (3, 4))

def macro_auc(P, T):
    T = T >= 0.5
    a = [roc_auc_score(T[:, k], P[:, k]) for k in range(12) if 0 < T[:, k].sum() < len(T)]
    return float(np.mean(a))

@torch.no_grad()
def predict(model, idx):
    model.eval(); out = []
    for x, m, _ in DataLoader(DS(idx, False), batch_size=8, num_workers=4):
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(model(prep(x.to(dev)), m.to(dev)).float().sigmoid().cpu())
    return torch.cat(out).numpy()

# split: gold-58 never trained on (true test); 15% of the rest = proxy val
rng = np.random.RandomState(1)
nong = np.where(~gold)[0]; rng.shuffle(nong)
nv = int(0.15 * len(nong)); val_idx, tr_idx = nong[:nv], nong[nv:]
gold_idx = np.where(gold)[0]
if SMOKE: tr_idx, val_idx = tr_idx[:SMOKE], val_idx[:SMOKE // 4]
print('train', len(tr_idx), '| val', len(val_idx), '| gold', len(gold_idx))

model = Net().to(dev).to(memory_format=torch.channels_last)
dl = DataLoader(DS(tr_idx, True), batch_size=BS, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
best = 0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tl = 0
    for it, (x, m, y) in enumerate(dl):
        x, m, y = aug(prep(x.to(dev))), m.to(dev), y.to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            logit = model(x, m)
        loss = F.binary_cross_entropy_with_logits(logit.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item()
        if it % 100 == 0: print(f'ep{ep} it{it}/{len(dl)} loss {tl/(it+1):.4f} {time.time()-t0:.0f}s', flush=True)
    pv = macro_auc(predict(model, val_idx), Y[val_idx])
    pg = macro_auc(predict(model, gold_idx), G[gold_idx])
    print(f'=== ep{ep} loss {tl/len(dl):.4f} | proxy val AUC {pv:.3f} | GOLD-58 AUC {pg:.3f} | {time.time()-t0:.0f}s')
    if pv > best: best = pv; torch.save(model.state_dict(), '/kaggle/working/model.pt')